<a href="https://colab.research.google.com/github/vitor-michelucci/postech-bb-tech-challenge-fase-3/blob/main/01_analise_churn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# POSTECH – BB | Tech Challenge Fase 3
## Análise de churn bancário
Objetivo: homologar consultas SQL no Colab antes da implementação em S3 e Athena.

**Etapa inicial:** carregar e validar a base. Execute as células em ordem.

## 1. Ambiente

In [ ]:
%pip -q install duckdb

In [ ]:
import pandas as pd
import duckdb
from google.colab import files
from IPython.display import display
print('DuckDB:', duckdb.__version__)

## 2. Carregamento local
Selecione o CSV original. Não publique dados bancários no GitHub sem autorização.

In [ ]:
from io import BytesIO

uploaded = files.upload()

# Validação da quantidade de arquivos enviados
assert len(uploaded) == 1, (
    'Selecione apenas um arquivo CSV.'
)

# Identificação automática do nome do arquivo
filename = next(iter(uploaded))

# Validação da extensão
assert filename.lower().endswith('.csv'), (
    'O arquivo selecionado deve estar no formato CSV.'
)

# Carregamento do dataset
df = pd.read_csv(BytesIO(uploaded[filename]))

print(f'Arquivo carregado: {filename}')
print('Dimensões:', df.shape)

display(df.head())

## 3. Validação da qualidade

In [ ]:
expected = ['customer_id','credit_score','country','gender','age','tenure','balance','products_number','credit_card','active_member','estimated_salary','churn']
assert list(df.columns) == expected, 'Estrutura inesperada'
display(pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isna().sum(), 'distintos': df.nunique()}))
print('Duplicatas:', df.duplicated().sum())
print('IDs duplicados:', df.customer_id.duplicated().sum())
for col in ['churn','active_member','credit_card']:
    assert df[col].isin([0,1]).all(), f'Valores inválidos: {col}'
assert df.isna().sum().sum() == 0
assert df.customer_id.is_unique

## 4. Conexão SQL e teste inicial
A tabela existe apenas na sessão atual do Colab.

In [ ]:
con = duckdb.connect(':memory:')
con.register('source_df', df)
con.execute('CREATE TABLE customers AS SELECT * FROM source_df')
display(con.sql('SELECT COUNT(*) AS total_clientes, SUM(churn) AS total_churn, ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct FROM customers').df())

In [ ]:
display(con.sql("SHOW TABLES").df())

In [ ]:
display(con.sql("DESCRIBE customers").df())

# 5. Análise exploratória de dados

## 5.1. Quantidade de clientes

Objetivo: identificar a quantidade total de registros
e de clientes distintos presentes na base de dados.

In [ ]:
query_01 = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT customer_id) AS total_clientes
FROM customers
"""

resultado_01 = con.sql(query_01).df()

display(resultado_01)

## 5.2. Taxa de churn total

Objetivo: mensurar a proporção de clientes que
deixaram a instituição financeira.

In [ ]:
query_02 = """
SELECT
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
"""

resultado_02 = con.sql(query_02).df()

display(resultado_02)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

valores = [
    resultado_02['clientes_retidos'].iloc[0],
    resultado_02['clientes_churn'].iloc[0]
]

plt.pie(
    valores,
    labels=['Permaneceram', 'Saíram'],
    autopct='%1.2f%%',
    colors=['#2E7D32', '#C62828'],
    startangle=90,
    wedgeprops={'width': 0.4, 'edgecolor': 'white'}
)

plt.title('Distribuição de Churn dos Clientes')
plt.tight_layout()
plt.show()

## 5.3. Taxa de churn por país

Objetivo: identificar diferenças geográficas na
taxa de saída dos clientes e avaliar quais mercados
apresentam maior incidência de churn.

In [ ]:
query_03 = """
SELECT
    country AS pais,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
GROUP BY country
ORDER BY taxa_churn_pct DESC
"""

resultado_03 = con.sql(query_03).df()

display(resultado_03)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

taxa_geral = resultado_02['taxa_churn_pct'].iloc[0]

cores = [
    '#C62828' if taxa > taxa_geral else '#2E7D32'
    for taxa in resultado_03['taxa_churn_pct']
]

barras = ax.bar(
    resultado_03['pais'],
    resultado_03['taxa_churn_pct'],
    color=cores
)

ax.axhline(
    taxa_geral,
    color='gray',
    linestyle='--',
    label=f'Churn geral: {taxa_geral:.2f}%'
)

ax.bar_label(
    barras,
    fmt='%.2f%%',
    padding=4
)

ax.set_title('Taxa de Churn por País')
ax.set_xlabel('País')
ax.set_ylabel('Taxa de churn (%)')
ax.set_ylim(0, resultado_03['taxa_churn_pct'].max() * 1.2)

ax.legend()

plt.tight_layout()
plt.show()

## 5.4. Taxa de churn por gênero

Objetivo: avaliar se existem diferenças nas taxas
de churn entre os gêneros e verificar a significância
estatística dessa associação por meio do teste
qui-quadrado de independência.

In [ ]:
query_04 = """
SELECT
    gender AS genero,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
GROUP BY gender
ORDER BY taxa_churn_pct DESC
"""

resultado_04 = con.sql(query_04).df()

display(resultado_04)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

taxa_geral = resultado_02['taxa_churn_pct'].iloc[0]

cores = [
    '#C62828' if taxa > taxa_geral else '#2E7D32'
    for taxa in resultado_04['taxa_churn_pct']
]

barras = ax.bar(
    resultado_04['genero'],
    resultado_04['taxa_churn_pct'],
    color=cores,
    width=0.5
)

ax.axhline(
    taxa_geral,
    color='gray',
    linestyle='--',
    label=f'Churn geral: {taxa_geral:.2f}%'
)

ax.bar_label(barras, fmt='%.2f%%', padding=4)

ax.set_title('Taxa de Churn por Gênero')
ax.set_xlabel('Gênero')
ax.set_ylabel('Taxa de churn (%)')
ax.set_ylim(0, resultado_04['taxa_churn_pct'].max() * 1.25)

ax.legend()

plt.tight_layout()
plt.show()

### 5.4.1. Teste qui-quadrado de independência

H0: gênero e churn são independentes.

H1: existe associação entre gênero e churn.

Nível de significância: 5%.

In [ ]:
query_04_contingencia = """
SELECT
    gender AS genero,
    SUM(CASE WHEN churn = 0 THEN 1 ELSE 0 END) AS retidos,
    SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) AS churn
FROM customers
GROUP BY gender
ORDER BY gender
"""

contingencia = con.sql(query_04_contingencia).df()

contingencia = contingencia.set_index('genero')

display(contingencia)

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_valor, gl, frequencias_esperadas = chi2_contingency(
    contingencia,
    correction=False
)

alpha = 0.05

print(f'Estatística qui-quadrado: {chi2:.4f}')
print(f'Graus de liberdade: {gl}')
print(f'P-valor: {p_valor:.6e}')
print(f'Nível de significância: {alpha:.2%}')

if p_valor < alpha:
    print('\nDecisão: rejeitar H0.')
    print('Existe associação estatisticamente significativa.')
else:
    print('\nDecisão: não rejeitar H0.')
    print('Não há evidência suficiente de associação.')

In [ ]:
esperadas = pd.DataFrame(
    frequencias_esperadas,
    index=contingencia.index,
    columns=contingencia.columns
)

display(esperadas.round(2))

print(
    'Menor frequência esperada:',
    round(frequencias_esperadas.min(), 2)
)

print(
    'Todas as frequências esperadas >= 5:',
    (frequencias_esperadas >= 5).all()
)

In [ ]:
import numpy as np

n = contingencia.to_numpy().sum()

r, c = contingencia.shape

v_cramer = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print(f'V de Cramér: {v_cramer:.4f}')

### Interpretação dos resultados

A taxa de churn entre clientes do gênero feminino foi
de 25,07%, enquanto entre clientes do gênero masculino
foi de 16,46%.

O teste qui-quadrado de independência apresentou
estatística de 113,4491 e p-valor de aproximadamente
1,72 × 10⁻²⁶.

Ao nível de significância de 5%, rejeita-se a hipótese
nula de independência entre gênero e churn.

Entretanto, o V de Cramér de 0,1065 indica que a
associação possui baixa magnitude.

Conclui-se que existe associação estatisticamente
significativa entre gênero e churn, mas que o gênero,
isoladamente, apresenta capacidade explicativa limitada.

Os resultados não demonstram causalidade e devem ser
avaliados conjuntamente com outras características
dos clientes.

## 5.5. Taxa de churn por faixa etária

Objetivo: identificar os segmentos etários com maior
incidência de churn e avaliar a relação entre idade
e permanência na instituição financeira.

In [ ]:
query_05 = """
WITH clientes_faixa AS (
    SELECT
        customer_id,
        churn,
        CASE
            WHEN age < 30 THEN 'Até 29 anos'
            WHEN age < 40 THEN '30 a 39 anos'
            WHEN age < 50 THEN '40 a 49 anos'
            WHEN age < 60 THEN '50 a 59 anos'
            ELSE '60 anos ou mais'
        END AS faixa_etaria,
        CASE
            WHEN age < 30 THEN 1
            WHEN age < 40 THEN 2
            WHEN age < 50 THEN 3
            WHEN age < 60 THEN 4
            ELSE 5
        END AS ordem
    FROM customers
)

SELECT
    faixa_etaria,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM clientes_faixa
GROUP BY faixa_etaria, ordem
ORDER BY ordem
"""

resultado_05 = con.sql(query_05).df()

display(resultado_05)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

taxa_geral = resultado_02['taxa_churn_pct'].iloc[0]

cores = [
    '#C62828' if taxa > taxa_geral else '#2E7D32'
    for taxa in resultado_05['taxa_churn_pct']
]

barras = ax.bar(
    resultado_05['faixa_etaria'],
    resultado_05['taxa_churn_pct'],
    color=cores
)

ax.axhline(
    taxa_geral,
    color='gray',
    linestyle='--',
    label=f'Churn geral: {taxa_geral:.2f}%'
)

ax.bar_label(
    barras,
    fmt='%.2f%%',
    padding=4
)

ax.set_title('Taxa de Churn por Faixa Etária')
ax.set_xlabel('Faixa etária')
ax.set_ylabel('Taxa de churn (%)')
ax.set_ylim(0, resultado_05['taxa_churn_pct'].max() * 1.2)

ax.legend()

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
query_05_complementar = """
WITH clientes_faixa AS (
    SELECT
        churn,
        CASE
            WHEN age < 30 THEN 'Até 29 anos'
            WHEN age < 40 THEN '30 a 39 anos'
            WHEN age < 50 THEN '40 a 49 anos'
            WHEN age < 60 THEN '50 a 59 anos'
            ELSE '60 anos ou mais'
        END AS faixa_etaria
    FROM customers
)

SELECT
    faixa_etaria,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct,
    ROUND(
        100.0 * SUM(churn) /
        (SELECT SUM(churn) FROM customers),
        2
    ) AS participacao_churn_pct
FROM clientes_faixa
GROUP BY faixa_etaria
ORDER BY clientes_churn DESC
"""

resultado_05_complementar = con.sql(
    query_05_complementar
).df()

display(resultado_05_complementar)

### Interpretação dos resultados

A análise por faixa etária evidencia diferenças
expressivas nas taxas de churn.

O segmento de 50 a 59 anos apresenta a maior taxa
de churn, atingindo 56,04%, aproximadamente 2,75
vezes a taxa geral da carteira.

Entretanto, a faixa de 40 a 49 anos concentra o
maior volume absoluto de saídas, com 806 clientes.

Em conjunto, os clientes entre 40 e 59 anos
representam 34,87% da carteira e concentram
63,48% do churn observado.

Recomenda-se aprofundar a investigação desse
segmento, avaliando saldo, quantidade de produtos,
nível de atividade e demais características
do relacionamento bancário.

Os resultados evidenciam uma associação não linear
entre idade e churn, sem estabelecer causalidade.

## Função auxiliar para visualização

Padronização dos gráficos de churn por segmento,
utilizando a taxa geral como referência comparativa.

In [ ]:
def plot_churn(
    dados,
    coluna_categoria,
    titulo,
    xlabel,
    figsize=(10, 5),
    rotacao=0
):
    fig, ax = plt.subplots(figsize=figsize)

    taxa_geral = resultado_02['taxa_churn_pct'].iloc[0]

    cores = [
        '#C62828' if taxa > taxa_geral else '#2E7D32'
        for taxa in dados['taxa_churn_pct']
    ]

    barras = ax.bar(
        dados[coluna_categoria].astype(str),
        dados['taxa_churn_pct'],
        color=cores
    )

    ax.axhline(
        taxa_geral,
        color='gray',
        linestyle='--',
        label=f'Churn geral: {taxa_geral:.2f}%'
    )

    ax.bar_label(
        barras,
        fmt='%.2f%%',
        padding=4
    )

    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Taxa de churn (%)')

    ax.set_ylim(
        0,
        max(dados['taxa_churn_pct'].max(), taxa_geral) * 1.2
    )

    ax.legend()

    plt.xticks(rotation=rotacao)
    plt.tight_layout()
    plt.show()

## 5.6. Relação entre saldo bancário e churn

Objetivo: avaliar se clientes com maiores saldos
bancários apresentam maior propensão ao churn,
considerando medidas de tendência central e
segmentação por faixas de saldo.

In [ ]:
query_06a = """
SELECT
    CASE
        WHEN churn = 0 THEN 'Retidos'
        ELSE 'Churn'
    END AS situacao,
    COUNT(*) AS total_clientes,
    ROUND(AVG(balance), 2) AS saldo_medio,
    ROUND(MEDIAN(balance), 2) AS saldo_mediano
FROM customers
GROUP BY churn
ORDER BY churn
"""

resultado_06a = con.sql(query_06a).df()

display(resultado_06a)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

resultado_06a.set_index('situacao')[
    ['saldo_medio', 'saldo_mediano']
].plot(
    kind='bar',
    ax=ax,
    color=['#1565C0', '#90CAF9'],
    rot=0
)

for barras in ax.containers:
    ax.bar_label(
        barras,
        fmt='%.0f',
        padding=3
    )

ax.set_title('Saldo Bancário por Situação de Churn')
ax.set_xlabel('Situação do cliente')
ax.set_ylabel('Saldo bancário')
ax.set_ylim(0, resultado_06a['saldo_mediano'].max() * 1.2)

ax.legend(['Saldo médio', 'Saldo mediano'])

plt.tight_layout()
plt.show()

In [ ]:
query_06b = """
WITH clientes_saldo AS (
    SELECT
        churn,
        CASE
            WHEN balance = 0 THEN 'Saldo zero'
            WHEN balance <= 50000 THEN 'Até 50 mil'
            WHEN balance <= 100000 THEN '50 a 100 mil'
            WHEN balance <= 150000 THEN '100 a 150 mil'
            WHEN balance <= 200000 THEN '150 a 200 mil'
            ELSE 'Acima de 200 mil'
        END AS faixa_saldo,
        CASE
            WHEN balance = 0 THEN 1
            WHEN balance <= 50000 THEN 2
            WHEN balance <= 100000 THEN 3
            WHEN balance <= 150000 THEN 4
            WHEN balance <= 200000 THEN 5
            ELSE 6
        END AS ordem
    FROM customers
)

SELECT
    faixa_saldo,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM clientes_saldo
GROUP BY faixa_saldo, ordem
ORDER BY ordem
"""

resultado_06b = con.sql(query_06b).df()

display(resultado_06b)

In [ ]:
plot_churn(
    dados=resultado_06b,
    coluna_categoria='faixa_saldo',
    titulo='Taxa de Churn por Faixa de Saldo',
    xlabel='Faixa de saldo bancário',
    figsize=(12, 6),
    rotacao=20
)

In [ ]:
query_06c = """
WITH clientes_saldo AS (
    SELECT
        churn,
        CASE
            WHEN balance = 0 THEN 'Saldo zero'
            ELSE 'Saldo positivo'
        END AS situacao_saldo
    FROM customers
)

SELECT
    situacao_saldo,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM clientes_saldo
GROUP BY situacao_saldo
ORDER BY taxa_churn_pct
"""

resultado_06c = con.sql(query_06c).df()

display(resultado_06c)

### Interpretação dos resultados

Os clientes que deixaram o banco apresentam saldo
médio de 91.108,54, aproximadamente 25,24% superior
ao saldo médio dos clientes retidos.

Clientes com saldo positivo apresentam taxa de
churn de 24,08%, contra 13,82% dos clientes com
saldo zero.

A maior taxa de churn foi observada na faixa
acima de 200 mil (55,88%). Entretanto, esse grupo
possui apenas 34 clientes.

A faixa de 100 a 150 mil concentra 987 saídas,
representando 48,45% do churn total.

Conclui-se que existe associação entre saldo
bancário e churn, mas essa relação não é linear.

Recomenda-se investigar os fatores de saída dos
clientes com maior relevância financeira, sem
atribuir causalidade exclusivamente ao saldo.

## 5.7. Relação entre quantidade de produtos e churn

Objetivo: avaliar a associação entre a quantidade
de produtos bancários e a taxa de churn,
identificando possíveis relações não lineares
entre o relacionamento comercial e a retenção.

In [ ]:
query_07 = """
SELECT
    products_number AS quantidade_produtos,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
GROUP BY products_number
ORDER BY products_number
"""

resultado_07 = con.sql(query_07).df()

display(resultado_07)

In [ ]:
plot_churn(
    dados=resultado_07,
    coluna_categoria='quantidade_produtos',
    titulo='Taxa de Churn por Quantidade de Produtos',
    xlabel='Quantidade de produtos bancários',
    figsize=(9, 5)
)

### Interpretação dos resultados

A relação entre quantidade de produtos e churn
apresenta comportamento não linear.

Clientes com dois produtos possuem a menor taxa
de churn (7,58%), enquanto clientes com apenas
um produto apresentam taxa de 27,71%.

Clientes com três e quatro produtos apresentam
taxas de 82,71% e 100%, respectivamente.
Entretanto, esses segmentos possuem populações
consideravelmente menores.

Os clientes com apenas um produto concentram
69,17% de todas as saídas registradas.

Recomenda-se investigar a adequação dos produtos
às necessidades dos clientes, a qualidade do
relacionamento comercial e as oportunidades
de retenção dos clientes com apenas um produto.

Os resultados não permitem concluir que a
contratação de um segundo produto, isoladamente,
cause redução do churn.

## 5.8. Relação entre atividade do cliente e churn

Objetivo: avaliar se clientes ativos apresentam
menor taxa de churn em comparação aos clientes
inativos, considerando o engajamento como uma
possível dimensão do relacionamento bancário.

In [ ]:
query_08 = """
SELECT
    CASE
        WHEN active_member = 1 THEN 'Ativo'
        ELSE 'Inativo'
    END AS situacao_atividade,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
GROUP BY active_member
ORDER BY taxa_churn_pct DESC
"""

resultado_08 = con.sql(query_08).df()

display(resultado_08)

In [ ]:
plot_churn(
    dados=resultado_08,
    coluna_categoria='situacao_atividade',
    titulo='Taxa de Churn por Nível de Atividade',
    xlabel='Situação de atividade',
    figsize=(8, 5)
)

### Interpretação dos resultados

Clientes ativos apresentam taxa de churn de 14,27%,
enquanto clientes inativos apresentam taxa de 26,85%.

A taxa observada entre clientes ativos é
aproximadamente 46,86% inferior à taxa observada
entre clientes inativos.

Os clientes inativos representam 48,49% da
carteira e concentram 63,92% do churn total.

Os resultados sugerem uma associação relevante
entre atividade bancária e retenção de clientes.

Recomenda-se investigar estratégias de engajamento,
reativação de clientes e monitoramento de sinais
de redução da atividade bancária.

A associação observada não permite estabelecer
uma relação de causalidade.

### Análise complementar: produtos e atividade

Objetivo: identificar segmentos de clientes
com diferentes taxas de churn a partir da
combinação entre quantidade de produtos
bancários e situação de atividade.

In [ ]:
query_08_complementar = """
SELECT
    products_number AS quantidade_produtos,
    CASE
        WHEN active_member = 1 THEN 'Ativo'
        ELSE 'Inativo'
    END AS situacao_atividade,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM customers
GROUP BY products_number, active_member
ORDER BY products_number, active_member
"""

resultado_08_complementar = con.sql(
    query_08_complementar
).df()

display(resultado_08_complementar)

In [ ]:
import seaborn as sns

matriz_churn = resultado_08_complementar.pivot(
    index='quantidade_produtos',
    columns='situacao_atividade',
    values='taxa_churn_pct'
)

matriz_churn = matriz_churn[
    ['Inativo', 'Ativo']
]

plt.figure(figsize=(8, 6))

sns.heatmap(
    matriz_churn,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn_r',
    vmin=0,
    vmax=100,
    linewidths=1,
    cbar_kws={'label': 'Taxa de churn (%)'}
)

plt.title('Churn por Quantidade de Produtos e Atividade')
plt.xlabel('Situação de atividade')
plt.ylabel('Quantidade de produtos')

plt.tight_layout()
plt.show()

### Interpretação da análise cruzada

A combinação entre quantidade de produtos e
atividade permite identificar segmentos com
diferentes padrões de churn.

Clientes inativos com apenas um produto
apresentam taxa de churn de 36,65% e concentram
45,36% de todas as saídas registradas.

Em contrapartida, clientes ativos com dois
produtos apresentam taxa de churn de apenas
5,56%, correspondente a uma retenção de 94,44%.

Recomenda-se priorizar a investigação do
segmento de clientes inativos com um produto,
avaliando oportunidades de engajamento e
adequação da oferta de produtos bancários.

A análise cruzada permite identificar segmentos
prioritários, mas não estabelece causalidade
entre as características analisadas e o churn.

## 5.9. Relação entre score de crédito e churn

Objetivo: avaliar a associação entre o score de
crédito dos clientes e a taxa de churn, utilizando
medidas de tendência central, segmentação por
faixas e análise estatística complementar.

In [ ]:
query_09a = """
SELECT
    CASE
        WHEN churn = 0 THEN 'Retidos'
        ELSE 'Churn'
    END AS situacao,
    COUNT(*) AS total_clientes,
    ROUND(AVG(credit_score), 2) AS score_medio,
    ROUND(MEDIAN(credit_score), 2) AS score_mediano,
    ROUND(STDDEV_SAMP(credit_score), 2) AS desvio_padrao
FROM customers
GROUP BY churn
ORDER BY churn
"""

resultado_09a = con.sql(query_09a).df()

display(resultado_09a)

In [ ]:
query_09b = """
WITH clientes_score AS (
    SELECT
        churn,
        CASE
            WHEN credit_score < 500 THEN '350 a 499'
            WHEN credit_score < 600 THEN '500 a 599'
            WHEN credit_score < 700 THEN '600 a 699'
            WHEN credit_score < 800 THEN '700 a 799'
            ELSE '800 a 850'
        END AS faixa_score
    FROM customers
)

SELECT
    faixa_score,
    COUNT(*) AS total_clientes,
    SUM(churn) AS clientes_churn,
    COUNT(*) - SUM(churn) AS clientes_retidos,
    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct
FROM clientes_score
GROUP BY faixa_score
ORDER BY faixa_score
"""

resultado_09b = con.sql(query_09b).df()

display(resultado_09b)

In [ ]:
plot_churn(
    dados=resultado_09b,
    coluna_categoria='faixa_score',
    titulo='Taxa de Churn por Faixa de Score',
    xlabel='Faixa de score de crédito',
    figsize=(10, 5)
)

### 5.9.1. Teste qui-quadrado de independência

H0: faixa de score e churn são independentes.

H1: existe associação entre faixa de score e churn.

Nível de significância: 5%.

In [ ]:
contingencia_score = resultado_09b.set_index(
    'faixa_score'
)[['clientes_retidos', 'clientes_churn']]

display(contingencia_score)

chi2_score, p_score, gl_score, esperadas_score = (
    chi2_contingency(
        contingencia_score,
        correction=False
    )
)

n_score = contingencia_score.to_numpy().sum()

r, c = contingencia_score.shape

v_cramer_score = np.sqrt(
    chi2_score / (n_score * min(r - 1, c - 1))
)

print(f'Qui-quadrado: {chi2_score:.4f}')
print(f'Graus de liberdade: {gl_score}')
print(f'P-valor: {p_score:.6f}')
print(f'V de Cramér: {v_cramer_score:.4f}')

print(
    'Menor frequência esperada:',
    round(esperadas_score.min(), 2)
)

if p_score < 0.05:
    print('Decisão: rejeitar H0.')
else:
    print('Decisão: não rejeitar H0.')

In [ ]:
from scipy.stats import pointbiserialr

correlacao_score, p_correlacao_score = pointbiserialr(
    df['churn'],
    df['credit_score']
)

print(f'Correlação: {correlacao_score:.6f}')
print(f'P-valor: {p_correlacao_score:.6f}')

### Interpretação dos resultados

Os clientes retidos apresentam score médio de
651,85 pontos, enquanto os clientes que saíram
apresentam score médio de 645,35 pontos.

A maior taxa de churn foi observada na faixa
de 350 a 499 pontos (23,73%), enquanto a menor
foi observada na faixa de 800 a 850 (19,54%).

O teste qui-quadrado aplicado às faixas de score
apresentou p-valor de 0,129650, não permitindo
rejeitar a hipótese nula ao nível de 5%.

A correlação ponto-bisserial apresentou
coeficiente de -0,0271 e p-valor de 0,006738.

Embora a correlação seja estatisticamente
significativa, sua magnitude é desprezível.

Conclui-se que o score de crédito apresenta
associação marginal de baixa relevância prática
com o churn nesta base de dados.

Sua contribuição em uma eventual análise
multivariada deverá ser avaliada separadamente.

## 5.10. Relação entre salário estimado e churn

Objetivo: comparar o salário estimado dos clientes retidos e dos clientes que deixaram a instituição.

In [ ]:
query_10 = """
SELECT
    CASE
        WHEN churn = 0 THEN 'Retidos'
        ELSE 'Churn'
    END AS situacao,
    COUNT(*) AS total_clientes,
    ROUND(AVG(estimated_salary), 2) AS salario_medio,
    ROUND(MEDIAN(estimated_salary), 2) AS salario_mediano,
    ROUND(STDDEV_SAMP(estimated_salary), 2) AS desvio_padrao
FROM customers
GROUP BY churn
ORDER BY churn
"""

resultado_10 = con.sql(query_10).df()

display(resultado_10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

resultado_10.set_index('situacao')[
    ['salario_medio', 'salario_mediano']
].plot(
    kind='bar',
    ax=ax,
    color=['#1565C0', '#90CAF9'],
    rot=0
)

for barras in ax.containers:
    ax.bar_label(
        barras,
        fmt='%.0f',
        padding=3
    )

ax.set_title('Salário Estimado por Situação de Churn')
ax.set_xlabel('Situação do cliente')
ax.set_ylabel('Salário estimado')

ax.set_ylim(
    0,
    resultado_10['salario_mediano'].max() * 1.2
)

ax.legend(['Salário médio', 'Salário mediano'])

plt.tight_layout()
plt.show()

In [ ]:
dados_salario = df.copy()

dados_salario['situacao'] = (
    dados_salario['churn'].map({
        0: 'Retidos',
        1: 'Churn'
    })
)

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=dados_salario,
    x='situacao',
    y='estimated_salary',
    hue='situacao',
    palette={
        'Retidos': '#2E7D32',
        'Churn': '#C62828'
    },
    legend=False
)

plt.title('Distribuição do Salário Estimado por Churn')
plt.xlabel('Situação do cliente')
plt.ylabel('Salário estimado')

plt.tight_layout()
plt.show()

### 5.10.1. Teste t de Welch

H0: os salários médios dos clientes retidos e
dos clientes que saíram são iguais.

H1: os salários médios dos dois grupos são
diferentes.

Nível de significância: 5%.

In [ ]:
from scipy.stats import ttest_ind

salario_churn = df.loc[
    df['churn'] == 1,
    'estimated_salary'
]

salario_retidos = df.loc[
    df['churn'] == 0,
    'estimated_salary'
]

teste_salario = ttest_ind(
    salario_churn,
    salario_retidos,
    equal_var=False
)

t_salario = teste_salario.statistic
p_salario = teste_salario.pvalue

print(f'Estatística t: {t_salario:.6f}')
print(f'P-valor: {p_salario:.6f}')

if p_salario < 0.05:
    print('Decisão: rejeitar H0.')
else:
    print('Decisão: não rejeitar H0.')

### Interpretação dos resultados

O salário médio estimado dos clientes que
permaneceram no banco foi de 99.738,39,
enquanto o dos clientes que saíram foi de
101.465,68.

A diferença observada corresponde a
aproximadamente 1,73%.

O teste t de Welch apresentou estatística
de 1,203368 e p-valor de 0,228925.

Ao nível de significância de 5%, não se
rejeita a hipótese nula de igualdade das
médias populacionais.

Os resultados não fornecem evidência suficiente
de diferença salarial média entre os grupos.

Assim, o salário estimado, quando analisado
isoladamente, não apresenta evidência de
diferenciação relevante entre clientes retidos
e clientes que saíram da instituição.

# 6. Auditoria e validação dos resultados

Esta seção tem como objetivo verificar a integridade
e a consistência dos resultados produzidos pelas
consultas SQL.

São realizadas verificações automatizadas de
reconciliação da população, quantidade de churn,
taxas calculadas e resultados estatísticos.

In [ ]:
TOTAL_CLIENTES = 10000
TOTAL_CHURN = 2037
TOTAL_RETIDOS = 7963
TAXA_CHURN = 20.37

print('Indicadores de referência definidos.')

In [ ]:
assert len(df) == TOTAL_CLIENTES

assert df['customer_id'].nunique() == TOTAL_CLIENTES

assert df.isna().sum().sum() == 0

assert df.duplicated().sum() == 0

assert df['churn'].sum() == TOTAL_CHURN

assert (df['churn'] == 0).sum() == TOTAL_RETIDOS

assert np.isclose(
    df['churn'].mean() * 100,
    TAXA_CHURN
)

print('Todos os indicadores gerais foram validados.')

In [ ]:
consultas_auditoria = {
    'Q03 - País': resultado_03,
    'Q04 - Gênero': resultado_04,
    'Q05 - Faixa etária': resultado_05,
    'Q06 - Faixa de saldo': resultado_06b,
    'Q06 - Situação do saldo': resultado_06c,
    'Q07 - Produtos': resultado_07,
    'Q08 - Atividade': resultado_08,
    'Q08 - Produtos e atividade': resultado_08_complementar,
    'Q09 - Faixa de score': resultado_09b
}

registros_auditoria = []

for nome, dados in consultas_auditoria.items():

    total = int(dados['total_clientes'].sum())
    churn = int(dados['clientes_churn'].sum())

    taxa = round(100 * churn / total, 2)

    validacao = (
        total == TOTAL_CLIENTES
        and churn == TOTAL_CHURN
        and np.isclose(taxa, TAXA_CHURN)
    )

    registros_auditoria.append({
        'consulta': nome,
        'total_clientes': total,
        'total_churn': churn,
        'taxa_churn_pct': taxa,
        'status': 'OK' if validacao else 'ERRO'
    })

auditoria = pd.DataFrame(registros_auditoria)

display(auditoria)

In [ ]:
for nome, dados in consultas_auditoria.items():

    taxa_recalculada = (
        100.0
        * dados['clientes_churn']
        / dados['total_clientes']
    )

    assert np.allclose(
        dados['taxa_churn_pct'],
        taxa_recalculada,
        atol=0.005001,
        rtol=0
    ), f'Divergência na consulta: {nome}'

    print(f'{nome}: OK')

print('\nTodas as taxas foram validadas.')

In [ ]:
testes_auditoria = pd.DataFrame([
    {
        'teste': 'Qui-quadrado - Gênero',
        'estatistica': chi2,
        'p_valor': p_valor,
        'p_referencia': 1.7204149874840846e-26
    },
    {
        'teste': 'Qui-quadrado - Score',
        'estatistica': chi2_score,
        'p_valor': p_score,
        'p_referencia': 0.12964962905245636
    },
    {
        'teste': 'Ponto-bisserial - Score',
        'estatistica': correlacao_score,
        'p_valor': p_correlacao_score,
        'p_referencia': 0.006738213892192579
    },
    {
        'teste': 'Teste t de Welch - Salário',
        'estatistica': t_salario,
        'p_valor': p_salario,
        'p_referencia': 0.22892461305145792
    }
])

testes_auditoria['validado'] = np.isclose(
    testes_auditoria['p_valor'],
    testes_auditoria['p_referencia'],
    rtol=1e-6,
    atol=0
)

display(testes_auditoria)

assert testes_auditoria['validado'].all()

print('Todos os testes estatísticos foram validados.')

In [ ]:
assert (auditoria['status'] == 'OK').all()

assert testes_auditoria['validado'].all()

print('=' * 60)
print('AUDITORIA TÉCNICA CONCLUÍDA COM SUCESSO')
print('=' * 60)

print(f'Clientes: {TOTAL_CLIENTES:,}')
print(f'Churn: {TOTAL_CHURN:,}')
print(f'Taxa de churn: {TAXA_CHURN:.2f}%')

print(f'Consultas reconciliadas: {len(auditoria)}')
print(f'Testes estatísticos validados: {len(testes_auditoria)}')

print('=' * 60)

# 7. Conclusões e recomendações

## 7.1. Síntese dos resultados

A análise exploratória contemplou 10.000 clientes
bancários, dos quais 2.037 apresentaram churn,
correspondendo a uma taxa geral de 20,37%.

As dez perguntas propostas pelo Tech Challenge
foram respondidas utilizando consultas SQL
executadas no DuckDB, complementadas por
visualizações e testes estatísticos em Python.

Os resultados evidenciam diferenças nas taxas
de churn associadas principalmente ao perfil
demográfico, à localização geográfica e às
características do relacionamento bancário.

A quantidade de produtos e o nível de atividade
dos clientes apresentaram diferenças expressivas
nas taxas de churn entre os segmentos analisados.

Em contrapartida, o score de crédito e o salário
estimado apresentaram baixa capacidade de
diferenciação quando analisados isoladamente.

## 7.2. Recomendações de negócio

- Investigar oportunidades de reativação de
  clientes inativos.

- Avaliar a adequação dos produtos bancários
  às necessidades dos clientes.

- Investigar os fatores associados às maiores
  taxas de churn no mercado alemão.

- Avaliar estratégias de retenção de clientes
  com maior relevância financeira.

- Aprofundar a análise dos segmentos etários
  com maior incidência de churn.

## 7.3. Limitações do estudo

- Ausência de informações sobre o período
  de observação dos dados.

- Ausência de informações sobre a moeda
  dos valores financeiros.

- Segmentos com baixa representatividade.

- Análises predominantemente bivariadas.

- Testes estatísticos exploratórios com
  p-valores não ajustados para multiplicidade.

- Impossibilidade de estabelecer relações
  de causalidade a partir dos resultados.

## 7.4. Próximas etapas

Após a homologação das consultas SQL no
ambiente Google Colab, a solução será
implementada na AWS, utilizando Amazon S3
para armazenamento e Amazon Athena para
processamento analítico.

Os resultados obtidos na nuvem serão
comparados com os resultados homologados
neste notebook.

# 8. Informações do ambiente de execução

Registro das versões do Python e das bibliotecas
utilizadas no desenvolvimento, para fins de
reprodutibilidade e documentação técnica.

In [ ]:
import sys
import scipy
import matplotlib
import seaborn

versoes = {
    'Python': sys.version.split()[0],
    'Pandas': pd.__version__,
    'NumPy': np.__version__,
    'DuckDB': duckdb.__version__,
    'Matplotlib': matplotlib.__version__,
    'Seaborn': seaborn.__version__,
    'SciPy': scipy.__version__
}

ambiente_execucao = pd.DataFrame(
    list(versoes.items()),
    columns=['Componente', 'Versão']
)

display(ambiente_execucao)